In [1]:
# Install dependencies
%pip install numpy Pillow torch torchvision open_clip_torch xgboost scikit-learn transformers


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.5 MB 1.1 MB/s eta 0:00:02
   ----- ---------------------------------- 0.2/1.5 MB 2.2 MB/s eta 0:00:01
   ----------- ---------------------------- 0.5/1.5 MB 3.2 MB/s eta 0:00:01
   -------------- ------------------------- 0.5/1.5 MB 2.8 MB/s eta 0:00:01
   -------------- ------------------------- 0.6/1.5 MB 2.7 MB/s eta 0:00:01
   -------------- ------------------------- 0.6/1.5 MB 2.7 MB/s eta 0:00:01
   --------------- ------------------------ 0.6/1.5 MB 1.9 MB/s eta 0:00:01
   ---------------- ----------------------- 0.7/1.5 MB 1.8 MB/s eta 0:00:01
   ------------------ --------------------- 0.7/1.5 MB 1.8 MB/s eta 0:00:01
   ------------------ --------------------- 0.7/1.5 MB 1.8 MB/s eta 0:00:01
   ------------------ --------------------- 0.7/1.5 MB 1.8 MB/s eta 0:00:01
   ------------------ --------------------- 0.7/1.5 MB 1.8 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\sulta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import math
import numpy as np
from PIL import Image

import torch
from sklearn.model_selection import train_test_split
import xgboost as xgb


In [3]:
# Paths
ROOT_DIR = Path.cwd().parent  # BarSight/
INPUT_DIR = ROOT_DIR / 'Input_jsons'
IMAGES_DIR = ROOT_DIR / 'dataset' / 'images'
PRED_DIR = ROOT_DIR / 'Error_bar_prediction'
GT_DIR = ROOT_DIR / 'Error_bar_groundTruth'

PRED_DIR.mkdir(parents=True, exist_ok=True)

# Feature extraction settings
MODEL_TYPE = 'clip'  # 'clip' or 'dinov2'
PATCH_SIZE = 96      # square crop around each data point
BATCH_SIZE = 64
SEED = 42

# XGBoost settings
XGB_PARAMS = {
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 300,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'random_state': SEED,
}

print('Input files:', len(list(INPUT_DIR.glob('*.json'))))
print('Images:', len(list(IMAGES_DIR.glob('*.png'))))


Input files: 150
Images: 150


In [4]:
def load_feature_model(model_type: str):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if model_type == 'clip':
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        model.eval().to(device)
        return model, preprocess, device, 'clip'
    if model_type == 'dinov2':
        from transformers import AutoImageProcessor, AutoModel
        processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
        model = AutoModel.from_pretrained('facebook/dinov2-base')
        model.eval().to(device)
        return model, processor, device, 'dinov2'
    raise ValueError('MODEL_TYPE must be clip or dinov2')

def crop_patch(image: Image.Image, x: float, y: float, size: int) -> Image.Image:
    half = size // 2
    left = int(round(x)) - half
    top = int(round(y)) - half
    right = left + size
    bottom = top + size
    return image.crop((left, top, right, bottom))

def extract_features(model, preproc, device, model_kind, patches, batch_size=64):
    features = []
    for i in range(0, len(patches), batch_size):
        batch = patches[i:i + batch_size]
        if model_kind == 'clip':
            tensor = torch.stack([preproc(p) for p in batch]).to(device)
            with torch.no_grad():
                feats = model.encode_image(tensor)
                feats = feats / feats.norm(dim=-1, keepdim=True)
        else:
            inputs = preproc(images=batch, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                out = model(**inputs)
                feats = out.last_hidden_state[:, 0, :]
        features.append(feats.cpu().numpy())
    if not features:
        return np.zeros((0, 1), dtype=np.float32)
    return np.vstack(features)

def _point_key(line_name, pt):
    return (
        line_name,
        round(float(pt['data_point']['x']), 2),
        round(float(pt['data_point']['y']), 2),
    )

def index_points(error_bars):
    index = {}
    for line in error_bars:
        line_name = line.get('lineName', '')
        for pt in line.get('points', []):
            key = _point_key(line_name, pt)
            index[key] = pt
    return index


In [5]:
# Build training dataset from ground-truth
torch.manual_seed(SEED)
np.random.seed(SEED)

model, preproc, device, model_kind = load_feature_model(MODEL_TYPE)

X_rows = []
y_up = []
y_down = []

input_files = sorted(INPUT_DIR.glob('*.json'))
for input_path in input_files:
    gt_path = GT_DIR / (input_path.stem + '.json')
    if not gt_path.exists():
        continue

    data = json.loads(input_path.read_text(encoding='utf-8'))
    image_file = data.get('image_file')
    line_groups = data.get('data_points', [])

    img_path = IMAGES_DIR / image_file
    if not img_path.exists():
        continue

    image = Image.open(img_path).convert('RGB')
    w, h = image.size

    gt = json.loads(gt_path.read_text(encoding='utf-8'))
    gt_index = index_points(gt.get('error_bars', []))

    patches = []
    meta = []
    for line in line_groups:
        line_name = line.get('lineName', '')
        for pt in line.get('points', []):
            key = (line_name, round(float(pt['x']), 2), round(float(pt['y']), 2))
            gt_pt = gt_index.get(key)
            if gt_pt is None:
                continue
            patches.append(crop_patch(image, pt['x'], pt['y'], PATCH_SIZE))
            meta.append((pt['x'], pt['y'], w, h, gt_pt))

    if not patches:
        continue

    feats = extract_features(model, preproc, device, model_kind, patches, BATCH_SIZE)

    for feat, (x, y, w, h, gt_pt) in zip(feats, meta):
        x_norm = float(x) / float(w)
        y_norm = float(y) / float(h)

        gu = float(gt_pt['upper_error_bar']['y'])
        gl = float(gt_pt['lower_error_bar']['y'])

        # Predict deltas (positive) from the data point
        delta_up = float(y) - gu
        delta_down = gl - float(y)

        X_rows.append(np.concatenate([np.array([x_norm, y_norm], dtype=np.float32), feat.astype(np.float32)]))
        y_up.append(delta_up)
        y_down.append(delta_down)

X = np.vstack(X_rows) if X_rows else np.zeros((0, 1), dtype=np.float32)
y_up = np.array(y_up, dtype=np.float32)
y_down = np.array(y_down, dtype=np.float32)

print('Training rows:', X.shape[0], 'Feature dim:', X.shape[1] if X.size else 0)


C:\Users\sulta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sulta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sulta\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co

Training rows: 4839 Feature dim: 514


In [6]:
# Train XGBoost regressors
if X.shape[0] == 0:
    raise RuntimeError('No training data found. Check GT/Input paths.')

X_train, X_val, y_up_train, y_up_val, y_down_train, y_down_val = train_test_split(
    X, y_up, y_down, test_size=0.2, random_state=SEED
)

up_model = xgb.XGBRegressor(**XGB_PARAMS)
down_model = xgb.XGBRegressor(**XGB_PARAMS)

up_model.fit(X_train, y_up_train)
down_model.fit(X_train, y_down_train)

up_val_pred = up_model.predict(X_val)
down_val_pred = down_model.predict(X_val)

mae_up = float(np.mean(np.abs(y_up_val - up_val_pred)))
mae_down = float(np.mean(np.abs(y_down_val - down_val_pred)))
{'val_mae_up': mae_up, 'val_mae_down': mae_down}


{'val_mae_up': 13.20608139038086, 'val_mae_down': 14.288208961486816}

In [7]:
# Run prediction for all input.json files
input_files = sorted(INPUT_DIR.glob('*.json'))
print(f'Found {len(input_files)} input files')

for idx, input_path in enumerate(input_files, start=1):
    data = json.loads(input_path.read_text(encoding='utf-8'))
    image_file = data.get('image_file')
    line_groups = data.get('data_points', [])

    img_path = IMAGES_DIR / image_file
    if not img_path.exists():
        print(f'[skip] image not found for {input_path.name}')
        continue

    image = Image.open(img_path).convert('RGB')
    w, h = image.size

    all_points = []
    patches = []
    for line in line_groups:
        line_name = line.get('lineName', '')
        for pt in line.get('points', []):
            patches.append(crop_patch(image, pt['x'], pt['y'], PATCH_SIZE))
            all_points.append((line_name, pt['x'], pt['y']))

    if patches:
        feats = extract_features(model, preproc, device, model_kind, patches, BATCH_SIZE)
    else:
        feats = np.zeros((0, X.shape[1] - 2), dtype=np.float32)

    points_output = {}
    for feat, (line_name, x, y) in zip(feats, all_points):
        x_norm = float(x) / float(w)
        y_norm = float(y) / float(h)
        feats_row = np.concatenate([np.array([x_norm, y_norm], dtype=np.float32), feat.astype(np.float32)])

        delta_up = float(up_model.predict(feats_row.reshape(1, -1))[0])
        delta_down = float(down_model.predict(feats_row.reshape(1, -1))[0])

        up_y = float(y) - max(0.0, delta_up)
        down_y = float(y) + max(0.0, delta_down)

        points_output.setdefault(line_name, []).append({
            'data_point': {'x': float(x), 'y': float(y)},
            'upper_error_bar': {'x': float(x), 'y': float(up_y)},
            'lower_error_bar': {'x': float(x), 'y': float(down_y)},
        })

    output = {
        'image_file': image_file,
        'error_bars': [
            {'lineName': ln, 'points': pts} for ln, pts in points_output.items()
        ],
    }

    (PRED_DIR / (input_path.stem + '.json')).write_text(json.dumps(output, indent=2), encoding='utf-8')

    if idx % 50 == 0:
        print(f'Predicted {idx}/{len(input_files)}')

print('Prediction done.')


Found 150 input files
Predicted 50/150
Predicted 100/150
Predicted 150/150
Prediction done.


In [11]:
# Evaluation settings
PIX_TOL = 3.0   # pixel tolerance for correct detection

def within_tol(a, b, tol):
    return abs(a - b) <= tol

tp = 0
fp = 0
fn = 0
abs_err_up = []
abs_err_down = []

gt_files = sorted(GT_DIR.glob('*.json'))
for gt_path in gt_files:
    pred_path = PRED_DIR / gt_path.name
    if not pred_path.exists():
        gt = json.loads(gt_path.read_text(encoding='utf-8'))
        gt_index = index_points(gt.get('error_bars', []))
        fn += len(gt_index)
        continue

    gt = json.loads(gt_path.read_text(encoding='utf-8'))
    pred = json.loads(pred_path.read_text(encoding='utf-8'))

    gt_index = index_points(gt.get('error_bars', []))
    pred_index = index_points(pred.get('error_bars', []))

    matched_pred_keys = set()
    for key, g in gt_index.items():
        p = pred_index.get(key)
        if p is None:
            fn += 1
            continue

        matched_pred_keys.add(key)

        gu = float(g['upper_error_bar']['y'])
        gl = float(g['lower_error_bar']['y'])
        pu = float(p['upper_error_bar']['y'])
        pl = float(p['lower_error_bar']['y'])

        abs_err_up.append(abs(gu - pu))
        abs_err_down.append(abs(gl - pl))

        ok = within_tol(gu, pu, PIX_TOL) and within_tol(gl, pl, PIX_TOL)
        if ok:
            tp += 1
        else:
            fp += 1

    extra = len(pred_index) - len(matched_pred_keys)
    if extra > 0:
        fp += extra

total = tp + fp + fn
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
accuracy = tp / total if total else 0.0

mae_up = float(np.mean(abs_err_up)) if abs_err_up else 0.0
mae_down = float(np.mean(abs_err_down)) if abs_err_down else 0.0

confusion_matrix = [[tp, fp], [fn, 0]]

{
    'points_total': total,
    'points_correct_within_tol': tp,
    'points_incorrect': fp,
    'points_missing': fn,
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'mae_upper': mae_up,
    'mae_lower': mae_down,
    'pixel_tolerance': PIX_TOL,
    'confusion_matrix': confusion_matrix,
}


{'points_total': 4839,
 'points_correct_within_tol': 3894,
 'points_incorrect': 945,
 'points_missing': 0,
 'accuracy': 0.8047117172969622,
 'precision': 0.8047117172969622,
 'recall': 1.0,
 'f1': 0.8917897629680522,
 'mae_upper': 3.2088008722680397,
 'mae_lower': 3.384156437614227,
 'pixel_tolerance': 3.0,
 'confusion_matrix': [[3894, 945], [0, 0]]}